<a href="https://colab.research.google.com/github/tjdux/Introduction-to-Machine-Learning-with-Python/blob/main/07_2_%ED%85%8D%EC%8A%A4%ED%8A%B8_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EB%8B%A4%EB%A3%A8%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# 노트북이 코랩에서 실행 중인지 체크합니다.
import os
import sys
if 'google.colab' in sys.modules and not os.path.isdir('mglearn'):
    # mglearn을 다운받고 압축을 풉니다.
    !wget -q -O mglearn.tar.gz https://bit.ly/mglearn-tar-gz
    !tar -xzf mglearn.tar.gz
    !wget -q -O data.tar.gz https://bit.ly/data-tar-gz
    !tar -xzf data.tar.gz
    # 나눔 폰트를 설치합니다.
    !sudo apt-get -qq -y install fonts-nanum
    import matplotlib.font_manager as fm
    font_files = fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum'])
    for fpath in font_files:
        fm.fontManager.addfont(fpath)

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fonts-nanum.
(Reading database ... 117528 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...


In [9]:
import sklearn
from preamble import *
import matplotlib

# 나눔 폰트를 사용합니다.
matplotlib.rc('font', family='NanumBarunGothic')
matplotlib.rcParams['axes.unicode_minus'] = False

/content/mglearn/datasets.py:31: SyntaxWarning: invalid escape sequence '\s'
  raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)


## 01 예제 애플리케이션: 영화 리뷰 감성 분석
- 데이터셋: 1-10점 중 7점 이상은 "양성", 4점 이하는 "음성"인 이진 분류 데이터 (중간은 포함하지 않음)

In [1]:
# 데이터 다운
!wget -nc http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz -P data
!tar xzf data/aclImdb_v1.tar.gz --skip-old-files -C data

--2026-01-08 02:33:23--  http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
Resolving ai.stanford.edu (ai.stanford.edu)... 171.64.68.10
Connecting to ai.stanford.edu (ai.stanford.edu)|171.64.68.10|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 84125825 (80M) [application/x-gzip]
Saving to: ‘data/aclImdb_v1.tar.gz’

aclImdb_v1.tar.gz   100%[===================>]  80.23M  13.9MB/s    in 8.6s    

2026-01-08 02:33:32 (9.29 MB/s) - ‘data/aclImdb_v1.tar.gz’ saved [84125825/84125825]



In [3]:
!find ./data -type d

./data
./data/aclImdb
./data/aclImdb/test
./data/aclImdb/test/neg
./data/aclImdb/test/pos
./data/aclImdb/train
./data/aclImdb/train/neg
./data/aclImdb/train/unsup
./data/aclImdb/train/pos


In [4]:
# 사용하지 않는 폴더 삭제
!rm -r data/aclImdb/train/unsup
!find ./data -type d

./data
./data/aclImdb
./data/aclImdb/test
./data/aclImdb/test/neg
./data/aclImdb/test/pos
./data/aclImdb/train
./data/aclImdb/train/neg
./data/aclImdb/train/pos


In [5]:
from sklearn.datasets import load_files

reviews_train = load_files("data/aclImdb/train/")
text_train, y_train = reviews_train.data, reviews_train.target
print(f"text_train의 타입: {type(text_train)}")
print(f"text_train의 길이: {len(text_train)}")
print(f"text_train[6]:\n{text_train[6]}")

text_train의 타입: <class 'list'>
text_train의 길이: 25000
text_train[6]:
b"This movie has a special way of telling the story, at first i found it rather odd as it jumped through time and I had no idea whats happening.<br /><br />Anyway the story line was although simple, but still very real and touching. You met someone the first time, you fell in love completely, but broke up at last and promoted a deadly agony. Who hasn't go through this? but we will never forget this kind of pain in our life. <br /><br />I would say i am rather touched as two actor has shown great performance in showing the love between the characters. I just wish that the story could be a happy ending."


In [6]:
# HTML 태그 삭제
text_train = [doc.replace(b"<br />", b" ") for doc in text_train]

In [10]:
# 같은 비율의 양성과 음성 레이블
print(f"클래스별 샘플 수 (훈련 데이터): {np.bincount(y_train)}")

클래스별 샘플 수 (훈련 데이터): [12500 12500]


In [12]:
reviews_test = load_files("data/aclImdb/test/")
text_test, y_test = reviews_test.data, reviews_test.target
print(f"테스트 데이터의 문서 수: {len(text_test)}")
print(f"클래스별 샘플 수 (테스트 데이터): {np.bincount(y_test)}")
text_test = [doc.replace(b"<br />", b" ") for doc in text_test]

테스트 데이터의 문서 수: 25000
클래스별 샘플 수 (테스트 데이터): [12500 12500]


## 02 텍스트 데이터를 BOW로 표현하기
- BOW(bag of words)
  - 가장 간단하지만 효과적이면서 널리 쓰이는 방법
  - 장, 문단, 문장, 서식 같은 입력 텍스트의 구조 대부분을 잃고, 각 단어가 이 말뭉치에 있는 텍스트에 얼마나 많이 나타나는지만 헤아림
  - 구조와 상관없이 단어의 출현 횟수만 셈
- BOW의 단계
  1. 토큰화(tokenization): 각 문서를 문서에 담긴 단어(토큰)로 나눔. 예를 들어 공백이나 구두점을 기준으로 분리
  2. 어휘 사전 구축: 모든 문서에 나타난 모든 단어의 어휘를 모으고 번호를 매김 (알파벳 순서)
  3. 인코딩: 어휘 사전의 단어가 문서마다 몇 번이나 나타나는지를 헤아림
- 출력은 각 문서에 나타난 단어의 횟수가 담긴 하나의 벡터

### 2.1 샘플 데이터에 BOW 적용하기

In [13]:
# 데이터 준비
bards_words = ["The fool doth think he is wise",
               "but the wise man knows himself to be a fool"]

In [14]:
from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer()
vect.fit(bards_words)

CountVectorizer()

In [15]:
# CountVectorizer.fit(): 토큰으로 나누고 어휘 사전을 구축
print(f"어휘 사전의 크기: {len(vect.vocabulary_)}")
print(f"어휘 사전의 내용:\n{vect.vocabulary_}")

어휘 사전의 크기: 13
어휘 사전의 내용:
{'the': 9, 'fool': 3, 'doth': 2, 'think': 10, 'he': 4, 'is': 6, 'wise': 12, 'but': 1, 'man': 8, 'knows': 7, 'himself': 5, 'to': 11, 'be': 0}


In [16]:
# CountVectorizer.transform(): BOW 표현을 만듦
bag_of_words = vect.transform(bards_words)
print(f"BOW: {repr(bag_of_words)}")

BOW: <Compressed Sparse Row sparse matrix of dtype 'int64'
	with 16 stored elements and shape (2, 13)>


- BOW 표현은 0이 아닌 값만 저장하는 희소 행렬로 저장 (특성 배열의 대부분의 원소가 0이라서 희소 행렬을 사용)
- 각각의 행은 하나의 데이터 포인트, 각 특성은 어휘 사전에 있는 각 단어에 대응

In [17]:
print(f"BOW의 밀집 표현:\n{bag_of_words.toarray()}")

BOW의 밀집 표현:
[[0 0 1 1 1 0 1 0 0 1 1 0 1]
 [1 1 0 1 0 1 0 1 1 1 0 1 1]]


- 각 단어의 출현 횟수는 0아니면 1
- 👉 두 문자열 모두 같은 단어를 두 개 이상 가지고 있지 않음
- 첫 번째 문자열 `The fool doth think he is wise`는 첫 번째 행으로 나타나며, 어휘 사전의 첫 번째 단어 `be`가 0번 나옴, 세 번째 단어 `doth`는 1번 나옴

### 2.2 영화 리뷰에 대한 BOW

In [18]:
vect = CountVectorizer().fit(text_train)
X_train = vect.transform(text_train)
print(f"X_train:\n{repr(X_train)}")

X_train:
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3431196 stored elements and shape (25000, 74849)>


In [20]:
feature_names = vect.get_feature_names_out()
print(f"특성 개수:{len(feature_names)}")
print(f"처음 20개 특성:\n{feature_names[:20]}")
print(f"20010에서 20030까지 특성:\n{feature_names[20010:20030]}")
print(f"매 2000번째 특성:\n{feature_names[::2000]}")

특성 개수:74849
처음 20개 특성:
['00' '000' '0000000000001' '00001' '00015' '000s' '001' '003830' '006'
 '007' '0079' '0080' '0083' '0093638' '00am' '00pm' '00s' '01' '01pm' '02']
20010에서 20030까지 특성:
['dratted' 'draub' 'draught' 'draughts' 'draughtswoman' 'draw' 'drawback'
 'drawbacks' 'drawer' 'drawers' 'drawing' 'drawings' 'drawl' 'drawled'
 'drawling' 'drawn' 'draws' 'draza' 'dre' 'drea']
매 2000번째 특성:
['00' 'aesir' 'aquarian' 'barking' 'blustering' 'bête' 'chicanery'
 'condensing' 'cunning' 'detox' 'draper' 'enshrined' 'favorit' 'freezer'
 'goldman' 'hasan' 'huitieme' 'intelligible' 'kantrowitz' 'lawful' 'maars'
 'megalunged' 'mostey' 'norrland' 'padilla' 'pincher' 'promisingly'
 'receptionist' 'rivals' 'schnaas' 'shunning' 'sparse' 'subset'
 'temptations' 'treatises' 'unproven' 'walkman' 'xylophonist']


- 희소 행렬의 고차원 데이터셋에서는 `LogisticRegression`같은 선형 모델의 성능이 가장 뛰어남

In [21]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

scores = cross_val_score(LogisticRegression(max_iter=1000), X_train, y_train,
                         n_jobs=-1)
print(f"교차 검증 평균 점수: {round(np.mean(scores), 2)}")

교차 검증 평균 점수: 0.88


In [22]:
from sklearn.model_selection import GridSearchCV

param_grid = {"C": [0.001, 0.01, 0.1, 1, 10]}
grid = GridSearchCV(LogisticRegression(max_iter=5000), param_grid, n_jobs=-1)
grid.fit(X_train, y_train)
print(f"최상의 교차 검증 점수: {round(grid.best_score_, 2)}")
print(f"최적의 매개변수: {grid.best_params_}")

최상의 교차 검증 점수: 0.89
최적의 매개변수: {'C': 0.1}


In [23]:
X_test = vect.transform(text_test)
print(f"테스트 점수: {round(grid.score(X_test, y_test), 2)}")

테스트 점수: 0.88


#### 2.2.1 단어 추출 방법 개선
- 숫자 같은 의미 없는 특성이 생성되는 것을 줄이는 방법: 적어도 두 개의 문서 (또는 다섯 개의 문서 등)에 나타난 토큰만을 사용

In [24]:
# min_df 매개변수로 토큰이 나타날 최소 문서 개수 지정
vect = CountVectorizer(min_df=5).fit(text_train)
X_train = vect.transform(text_train)
print(f"min_df로 제한한 X_train: {repr(X_train)}")

min_df로 제한한 X_train: <Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3354014 stored elements and shape (25000, 27271)>


In [25]:
feature_names = vect.get_feature_names_out()

print(f"처음 50개 특성:\n{feature_names[:50]}")
print(f"20010부터 20030까지 특성:\n{feature_names[20010:20030]}")
print(f"매 700번째 특성:\n{feature_names[::700]}")

처음 50개 특성:
['00' '000' '007' '00s' '01' '02' '03' '04' '05' '06' '07' '08' '09' '10'
 '100' '1000' '100th' '101' '102' '103' '104' '105' '107' '108' '10s'
 '10th' '11' '110' '112' '116' '117' '11th' '12' '120' '12th' '13' '135'
 '13th' '14' '140' '14th' '15' '150' '15th' '16' '160' '1600' '16mm' '16s'
 '16th']
20010부터 20030까지 특성:
['repentance' 'repercussions' 'repertoire' 'repetition' 'repetitions'
 'repetitious' 'repetitive' 'rephrase' 'replace' 'replaced' 'replacement'
 'replaces' 'replacing' 'replay' 'replayable' 'replayed' 'replaying'
 'replays' 'replete' 'replica']
매 700번째 특성:
['00' 'affections' 'appropriately' 'barbra' 'blurbs' 'butchered' 'cheese'
 'commitment' 'courts' 'deconstructed' 'disgraceful' 'dvds' 'eschews'
 'fell' 'freezer' 'goriest' 'hauser' 'hungary' 'insinuate' 'juggle'
 'leering' 'maelstrom' 'messiah' 'music' 'occasional' 'parking'
 'pleasantville' 'pronunciation' 'recipient' 'reviews' 'sas' 'shea'
 'sneers' 'steiger' 'swastika' 'thrusting' 'tvs' 'vampyre' 'western

In [27]:
grid = GridSearchCV(LogisticRegression(max_iter=5000), param_grid, n_jobs=-1)
grid.fit(X_train, y_train)
print(f"최상의 교차 검증 점수: {round(grid.best_score_, 2)}")

최상의 교차 검증 점수: 0.89


- 모델 성능은 높아지지 않았지만 특성의 개수가 줄어서 처리 속도가 빨라지고, 불필요한 특성이 없어져 모델을 이해하기 쉬워짐